# ⭐ STAR — Neuro-Symbolic AI
### *by [Reagent Systems](https://github.com/reagent-systems)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyFriendlyFox/star/blob/main/STAR_Colab.ipynb)

---

**STAR** is a neuro-symbolic AI research project that combines neural network learning with symbolic reasoning. This notebook runs both core systems:

| System | What it does |
|---|---|
| **Graph Transformer** | Learns to evaluate LISP S-expressions via a custom transformer over graph-structured data |
| **Neuro-Symbolic Prototype** | DistilBERT relation classifier + symbolic knowledge base with forward-chaining inference |

> **⚠️ Runtime Setup:** Before running, go to **Runtime → Change runtime type** and select:
> - **GPU: A100**
> - **High RAM** enabled
>
> This ensures enough VRAM and system memory for both training pipelines.

## 0 · GPU Check

In [ ]:
import torch, os, psutil

assert torch.cuda.is_available(), "No GPU detected — please enable A100 in Runtime settings."

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
ram_gb  = psutil.virtual_memory().total / 1e9

print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"RAM  : {ram_gb:.1f} GB")
print()

if "A100" in gpu_name:
    print("✅ A100 detected — optimal performance.")
else:
    print(f"⚠️  Got {gpu_name} instead of A100. Things will still work but may be slower.")

if ram_gb > 50:
    print("✅ High RAM enabled.")
else:
    print(f"⚠️  RAM is {ram_gb:.0f} GB — consider enabling High RAM in runtime settings.")

## 1 · Install Dependencies

In [ ]:
!pip install -q torch numpy datasets transformers tqdm networkx matplotlib pydot

## 2 · Clone the STAR Repository

In [ ]:
# Clone repo (skip if already present)
if not os.path.isdir("star"):
    !git clone https://github.com/ThyFriendlyFox/star.git
else:
    print("Repository already cloned.")

%cd star

---
# Part 1 · Graph Transformer

The Graph Transformer learns to **evaluate LISP S-expressions** (e.g. `(+ 1 2)` → `3`) by operating directly on their graph structure rather than treating them as flat token sequences.

Key ideas:
- S-expressions are parsed into directed acyclic graphs
- A transformer processes node embeddings with structure-aware attention
- The model learns symbolic computation from examples

Training includes an **overfit sanity check** on a small batch to verify the model can memorize before scaling up.

In [ ]:
# Train the Graph Transformer (~3000 steps on A100 ≈ a few minutes)
!python train.py --device cuda --steps 3000 --batch-size 32

---
# Part 2 · Neuro-Symbolic Prototype

This system combines a **neural relation classifier** with a **symbolic knowledge base**:

1. **Neural**: Fine-tunes DistilBERT to extract semantic relations from natural language sentences
2. **Symbolic**: Stores extracted relations in a structured knowledge base
3. **Reasoning**: Applies **forward-chaining inference** rules to derive new facts

The pipeline trains the classifier, builds the KB, runs inference, and exports everything.

In [ ]:
# Create output directory
!mkdir -p out checkpoints

# Run the full neuro-symbolic pipeline
!python prototypes/neuro_symbolic_vector_graph_prototype.py \
    --device cuda \
    --epochs 2 \
    --batch-size 16 \
    --export-json out/kb.json \
    --export-dot out/kb.dot \
    --trace-inference \
    --save-checkpoint checkpoints/semeval_re.pt

### Try a Custom Sentence

Use `--predict-only` to skip training and run inference on your own sentence. Edit the text below to try different inputs.

In [ ]:
# Try custom inference — change the sentence to whatever you like
!python prototypes/neuro_symbolic_vector_graph_prototype.py \
    --device cuda \
    --predict-only \
    --load-checkpoint checkpoints/semeval_re.pt \
    --infer-sentence "The <e1>earthquake</e1> caused the <e2>tsunami</e2>" \
    --predict-top-k 3

---
# Part 3 · Visualize the Knowledge Graph

Render the exported knowledge base as an interactive graph image. Nodes are entities, edges are relations — including any facts derived by forward chaining.

In [ ]:
# Generate the graph visualization
!python prototypes/view_kb_graph.py out/kb.json \
    --out out/preview.png \
    --show \
    --edge-labels

# Display inline
from IPython.display import Image, display
display(Image(filename="out/preview.png"))

---
## 📝 Notes & Next Steps

**Repository:** [github.com/ThyFriendlyFox/star](https://github.com/ThyFriendlyFox/star)

**What to explore next:**
- Modify `train.py` args to experiment with deeper models or longer training
- Add custom inference rules to the neuro-symbolic prototype
- Try different sentences and inspect the knowledge graph output
- Check `out/kb.json` for the raw knowledge base structure
- Open `out/kb.dot` in Graphviz for a different graph layout

---
*Built by [Reagent Systems](https://github.com/reagent-systems) · STAR — Neuro-Symbolic AI*